In [ ]:
!pip install -q nltk scikit-learn seaborn matplotlib transformers torch

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
reddit_df = pd.read_csv("Reddit_Data.csv")
twitter_df = pd.read_csv("Twitter_Data.csv")

print("Reddit Columns:", reddit_df.columns)
print("Twitter Columns:", twitter_df.columns)

Reddit Columns: Index(['clean_comment', 'category'], dtype='object')
Twitter Columns: Index(['clean_text', 'category'], dtype='object')


In [ ]:
# Reddit
reddit_df = reddit_df.rename(columns={
    'clean_comment': 'text',
    'category': 'label'
})

# Twitter
twitter_df = twitter_df.rename(columns={
    'clean_text': 'text',
    'category': 'label'
})

reddit_df = reddit_df[['text', 'label']]
twitter_df = twitter_df[['text', 'label']]

In [ ]:
df = pd.concat([reddit_df, twitter_df], ignore_index=True)
df.dropna(inplace=True)
df['label'] = df['label'].astype(int)

print(df.shape)
df.head()

(200118, 2)


,text,label
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return " ".join(tokens)

df['processed_text'] = df['text'].apply(preprocess)

In [ ]:
X = df['processed_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
vectorizer = TfidfVectorizer(max_features=6000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SGD Classifier": SGDClassifier(loss="log_loss")
}

In [ ]:
results = []

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, preds, average="weighted", zero_division=0),
        "F1-Score": f1_score(y_test, preds, average="weighted", zero_division=0)
    })

ml_results = pd.DataFrame(results)
ml_results

,Model,Accuracy,Precision,Recall,F1-Score
0,Naive Bayes,0.724190,0.750143,0.724190,0.714646
1,Logistic Regression,0.876599,0.877715,0.876599,0.875238
2,Linear SVM,0.886893,0.887840,0.886893,0.885947
3,Decision Tree,0.802718,0.801327,0.802718,0.801910
4,Random Forest,0.843219,0.843631,0.843219,0.839444
5,KNN,0.430242,0.664839,0.430242,0.354936
6,SGD Classifier,0.788877,0.801500,0.788877,0.778291


In [ ]:
label_map = {-1: 0, 0: 1, 1: 2}
reverse_map = {0: -1, 1: 0, 2: 1}

df['bert_label'] = df['label'].map(label_map)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize_data(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=128
    )

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'],
    df['bert_label'],
    test_size=0.2,
    random_state=42,
    stratify=df['bert_label']
)

train_encodings = tokenize_data(train_texts)
test_encodings = tokenize_data(test_texts)


In [ ]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, train_labels)
test_dataset = SentimentDataset(test_encodings, test_labels)


In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    logging_dir="./logs",
    save_strategy="no",
    report_to="none"   # 🔥 VERY IMPORTANT
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.124100,0.103264


Epoch,Training Loss,Validation Loss
1,0.124100,0.103264
2,0.069500,0.070298


TrainOutput(global_step=20012, training_loss=0.14100030150957735, metrics={'train_runtime': 7417.9365, 'train_samples_per_second': 43.164, 'train_steps_per_second': 2.698, 'total_flos': 2.1061439748873216e+16, 'train_loss': 0.14100030150957735, 'epoch': 2.0})

In [ ]:
preds = trainer.predict(test_dataset)
bert_preds = np.argmax(preds.predictions, axis=1)

print("BERT Accuracy:", accuracy_score(test_labels, bert_preds))
print("BERT Precision:", precision_score(test_labels, bert_preds, average="weighted"))
print("BERT Recall:", recall_score(test_labels, bert_preds, average="weighted"))
print("BERT F1:", f1_score(test_labels, bert_preds, average="weighted"))


BERT Accuracy: 0.9839096542074756
BERT Precision: 0.9839263613000319
BERT Recall: 0.9839096542074756
BERT F1: 0.9839132750928175
